In [61]:
import numpy as np
from firedrake import *
from firedrake.cython import dmcommon
from petsc4py import PETSc
import math

In [62]:
# parameters in SI units
t_end = 5.0  # time of simulation [s]
dt = 0.005  # time step [s]
g = 9.8  # gravitational acceleration
# water
Lx = 20.0  # length of the tank [m] in x-direction; needed for computing initial condition
Lz = 10.0  # height of the tank [m]; needed for computing initial condition
rho = 1000.0  # fluid density in kg/m^2 in 2D [water]
# solid parameters
#  - we use a sufficiently soft material to be able to see noticeable structural displacement
rho_B = 7700.0  # structure density in kg/m^2 in 2D
lam = 1e7  # N/m in 2D - first Lame constant
mu = 1e7  # N/m in 2D - second Lame constant

# these numbers must match the ones defined in the mesh file
fluid_id = 1  # fluid subdomain
structure_id = 2  # structure subdomain
bottom_id = 1  # structure bottom
top_id = 6  # fluid surface
interface_id = 9  # fluid-structure interface

L = Lz
T = L / math.sqrt(g * L)
t_end /= T
dt /= T
Lx /= L
Lz /= L
rho_B /= rho
lam /= g * rho * L
mu /= g * rho * L
rho = 1.0  # or equivalently rho /= rho

mesh = Mesh("/home/charlottecai/MSc_Project_new/L_domain.msh")
dim = 2

# Extract submesh
mesh_F = Submesh(mesh, dim, fluid_id,label_name=dmcommon.CELL_SETS_LABEL, name="fluid_mesh")
x_F = SpatialCoordinate(mesh_F)
n_F = FacetNormal(mesh_F)

mesh_S = Submesh(mesh, dim, structure_id, label_name=dmcommon.CELL_SETS_LABEL, name="solid_mesh")
x_S = SpatialCoordinate(mesh_S)
n_S = FacetNormal(mesh_S)

# free surface submesh
mesh_T = Submesh(mesh_F, dim - 1, top_id, name="free_surface_mesh",)
x_T = SpatialCoordinate(mesh_T)

# Function spaces
V_W = FunctionSpace(mesh_F, "CG", 1)
V_B = VectorFunctionSpace(mesh_S, "CG", 1)
V_T = FunctionSpace(mesh_T, "CG", 1)

mixed_V = V_W * V_B

# Measures
dx_F = Measure("dx", mesh_F)
dx_S = Measure("dx", mesh_S)
dx_T = Measure("dx", mesh_T)

# interface measures
ds_F = Measure("ds", mesh_F, intersect_measures=(Measure("ds", mesh_S),))
ds_S = Measure("ds", mesh_S, intersect_measures=(Measure("ds", mesh_F),))
dz_T = Measure("dx", mesh_T, intersect_measures=(Measure("ds", mesh_F),))

# fluid domain
phi = Function(V_W, name="phi")
trial_W = TrialFunction(V_W)
v_W = TestFunction(V_W)

# free surface
phi_f = Function(V_T, name="phi_f")
eta = Function(V_T, name="eta")
trial_T = TrialFunction(V_T)
v_T = TestFunction(V_T)

# Structure domain
X = Function(V_B, name="X")
U = Function(V_B, name="U")

trial_B = TrialFunction(V_B)
v_B = TestFunction(V_B)

# mixed domain
trial_f, trial_s = TrialFunctions(mixed_V)
v_f, v_s = TestFunctions(mixed_V)

tmp_f = Function(V_W)
tmp_s = Function(V_B)
result_mixed = Function(mixed_V)

# Carrier used to impose phi_f as a BC on the fluid-volume space
phi_f_on_F = Function(V_W, name="phi_f_on_F")

In [63]:
# Boundary conditions
#BC_phi_f = DirichletBC(mixed_V.sub(0), phi_f, top_id)
BC_phi_f = DirichletBC(mixed_V.sub(0), phi_f_on_F, top_id)
BC_bottom = DirichletBC(V_B, as_vector([0.0, 0.0]), bottom_id)
BC_bottom_mixed = DirichletBC(mixed_V.sub(1), as_vector([0.0, 0.0]), bottom_id)


# solvers
# 1. equations for phi at the free surface----update for free-surface submesh
a_phi_f = trial_T * v_T * dx_T
L_phi_f = (phi_f - dt * eta) * v_T * dx_T
LVP_phi_f = LinearVariationalProblem(a_phi_f, L_phi_f, phi_f)
LVS_phi_f = LinearVariationalSolver(LVP_phi_f)

# 2. equations for  X (beam displacement)
a_X = dot(trial_B, v_B) * dx_S
L_X = dot((X + dt * U), v_B) * dx_S

LVP_X = LinearVariationalProblem(a_X, L_X, X, bcs=[BC_bottom])
LVS_X = LinearVariationalSolver(LVP_X)

# 3. equations for phi at the fluid domain, U and eta
# elastic stiffness term
delX = nabla_grad(X)
delv_B = nabla_grad(v_s)
T_x_dv = lam * div(X) * div(v_s) + mu * inner(delX, delv_B + transpose(delv_B))
a_U = rho_B * dot(trial_s, v_s) * dx_S
L_U = (rho_B * dot(U, v_s) - dt * T_x_dv) * dx_S
a_phi = dot(grad(trial_f), grad(v_f)) * dx_F

# previous wrong try, delete later
#a_U += dot(v_s, n_S) * trial_f * ds_S(interface_id)
#L_U += dot(v_s, n_S) * phi * ds_S(interface_id)
#a_phi += -dot(n_F, trial_s) * v_f * ds_F(interface_id)
#a_U += dot(v_s, n_F) * trial_f * ds_F(interface_id)
#L_U += dot(v_s, n_F) * phi * ds_F(interface_id)
#a_phi += -dot(n_F, trial_s) * v_f * ds_F(interface_id)

# Each weak form includes the boundary terms generated by integration by parts over its own domain.
# -n_F=n_S
# or a_U += dot(v_s, n_F) * trial_f * ds_S(interface_id)
# or L_U += dot(v_s, n_F) * phi * ds_S(interface_id)
a_U += -dot(v_s, n_S) * trial_f * ds_S(interface_id)
L_U += -dot(v_s, n_S) * phi * ds_S(interface_id)

a_phi += -dot(n_F, trial_s) * v_f * ds_F(interface_id)

LVP_U_phi = LinearVariationalProblem(
    a_U + a_phi,
    L_U,
    result_mixed,
    bcs=[BC_phi_f, BC_bottom_mixed]
)

LVS_U_phi = LinearVariationalSolver(LVP_U_phi)

In [64]:
# eta
a_eta = trial_T * v_T * dx_T
L_eta = (eta * v_T * dx_T + dt * v_T * dot(grad(phi), n_F) * dz_T)

LVP_eta = LinearVariationalProblem(a_eta, L_eta, eta)
LVS_eta = LinearVariationalSolver(LVP_eta)

In [65]:
# initial condition
n_mode = 1
a = 0.0 * T / L ** 2
b = 5.0 * T / L ** 2
lambda_x = np.pi * n_mode / Lx
omega = np.sqrt(lambda_x * np.tanh(lambda_x * Lz))

phi_exact_expr = a * cos(lambda_x * x_F[0]) * cosh(lambda_x * x_F[1])
phi_f_exact_expr = a * cos(lambda_x * x_T[0]) * cosh(lambda_x * x_T[1])
eta_exact_expr = -omega * b * cos(lambda_x * x_T[0]) * cosh(lambda_x * Lz)

phi.interpolate(phi_exact_expr)
phi_f.interpolate(phi_f_exact_expr)
eta.interpolate(eta_exact_expr)

eta_exact = Function(V_T, name="eta_exact")
eta_exact.interpolate(eta_exact_expr)

phi_f_on_F.interpolate(phi_exact_expr)

Coefficient(WithGeometry(FunctionSpace(<firedrake.mesh.MeshTopology object at 0x742ba2a1b8e0>, FiniteElement('Lagrange', triangle, 1), name=None), Mesh(VectorElement(FiniteElement('Lagrange', triangle, 1), dim=2), 106553)), 206195)

In [66]:
# Output files

# to avoid saving data every time step
output_data_every_x_time_steps = 20

outfile_phi = VTKFile("results_pvd_submesh/phi.pvd")
#outfile_phi_f = VTKFile("results_pvd_submesh/phi_f.pvd")
#outfile_eta = VTKFile("results_pvd_submesh/eta.pvd")
#outfile_U = VTKFile("results_pvd_submesh/U.pvd")
#outfile_X = VTKFile("results_pvd_submesh/X.pvd")


def output_data():
    output_data.counter += 1

    if output_data.counter % output_data_every_x_time_steps != 0:
        return

    outfile_phi.write(phi)
    #outfile_phi_f.write(phi_f)
    #outfile_eta.write(eta)
    #outfile_U.write(U)
    #outfile_X.write(X)


output_data.counter = -1 # -1 to exclude counting print of initial state

In [67]:
# Time loop
output_data()

num_steps = int(round(t_end / dt))

for step in ProgressBar("Time step").iter(np.linspace(0, t_end, int(t_end/dt))):

    LVS_phi_f.solve()

    phi_f_on_F.interpolate(phi_f, allow_missing_dofs=True)

    LVS_U_phi.solve()

    tmp_f, tmp_s = result_mixed.subfunctions
    phi.assign(tmp_f)
    U.assign(tmp_s)

    LVS_eta.solve()

    LVS_X.solve()

    output_data()

Time step ▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣▣ 1000/1000 [0:01:09]
